# Πολυτροπική Ανάλυση Μουσικών Δεδομένων — Multimodal Music Analysis

**Μάθημα:** Τεχνικές Εξόρυξης Δεδομένων

**[ΟΝΟΜΑ ΦΟΙΤΗΤΗ] — sdi2200160**

In [ ]:
%pip install -q pandas numpy torch sentence-transformers transformers scikit-learn matplotlib seaborn wordcloud nltk rich

In [ ]:
from __future__ import annotations

import functools
import logging
import math
import pathlib
import tarfile
import typing
from dataclasses import dataclass

import matplotlib.axes
import matplotlib.figure
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rich.progress
import seaborn as sns
import sentence_transformers
import torch
from nltk.corpus import stopwords
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
from wordcloud import WordCloud

import nltk
nltk.download("stopwords", quiet=True)

%matplotlib inline

# === Configuration ===

DATA_DIR = pathlib.Path("data")
K = 5

## 1. Συλλογή Δεδομένων — Data Collection

We are given five raw data files (all tab-separated, keyed by Song ID):

| File | Description |
| ---- | ----------- |
| `id_mfcc_stats.tsv.bz2` | MFCC means (13) and covariance matrix (91 entries) per song |
| `processed_lyrics.tar.gz` | Preprocessed/stemmed lyrics, one text file per song |
| `id_genres.csv` | Comma-separated genre labels per song (multilabel) |
| `id_tags.csv` | Comma-separated user tags per song |
| `id_information.csv` | Song metadata: artist, song name, album |

### Design Choices

> **Multilabel genres**: The assignment suggests filtering to ~5,000–10,000 songs by selecting the top-5 genres as a *multiclass* problem (one genre per song). We instead treat genres as **multilabel** — a song tagged `"rock,indie rock"` belongs to *both* genres. This retains far more songs (~54,000 after intersection) and better reflects the reality that music rarely fits a single category. This choice propagates through the entire pipeline: encodings are binary vectors, not one-hot; evaluation metrics must account for multilabel structure.

> **Full feature intersection**: Rather than intersecting only the minimum three files (audio, lyrics, genre), we include **all five sources** — adding tags and song metadata. Songs with any missing value in any source are dropped. This gives us a richer representation at the cost of a slightly smaller intersection.

In [ ]:
type BinaryDataFrame = pd.DataFrame


class MusicSeries(pd.Series):

    @property
    def _constructor(self) -> type[typing.Self]:
        return self.__class__

    @classmethod
    def from_csv(cls, path: str | pathlib.Path) -> typing.Self:
        return cls(
            pd.read_csv(path,
                sep="\t",
                index_col=0,
                low_memory=False,
            ).squeeze()
        )

    @classmethod
    def from_tar(cls, path: str | pathlib.Path) -> typing.Self:
        records: dict[str, str] = {}

        with tarfile.open(path, "r:gz") as archive:
            for member in archive.getmembers():
                if not member.isfile():
                    continue

                song_id = pathlib.Path(member.name).stem
                file = archive.extractfile(member)

                if file is not None:
                    records[song_id] = file.read().decode("utf-8",
                        errors="replace",
                    )

        return cls(
            pd.Series(records, name="lyrics")
        )

    @functools.cached_property
    def encoding(self) -> BinaryDataFrame:
        return self.str.get_dummies(sep=",")

    def mask(self, genres: typing.Iterable[str], multi: bool = True) -> pd.Series:
        if multi: return self.encoding[genres].any(axis="columns")
        else: return self.isin(genres)

    def distribution(self, multi: bool = True) -> pd.Series:
        if multi: return self.encoding.sum(axis="index")
        else: return self.value_counts()

    def top_labels(self, k: int, multi: bool = True) -> pd.Index:
        if multi: return self.distribution(multi=True).sort_values(ascending=False).head(k).index
        else: return self.distribution(multi=False).head(k).index

    def top(self, k: int, multi: bool = True) -> pd.Series:
        return self[self.mask(self.top_labels(k, multi=multi), multi=multi)]


class MusicDataFrame(pd.DataFrame):

    @property
    def _constructor(self) -> type[typing.Self]:
        return self.__class__

    @classmethod
    def from_csv(cls, path: str | pathlib.Path) -> typing.Self:
        return cls(
            pd.read_csv(path,
                sep="\t",
                index_col=0,
                low_memory=False,
            )
        )

    def intersection(self, *attributes: pd.Series) -> pd.DataFrame:
        indices = self.index.intersection(
            pd.Index(set.intersection(*(set(attribute.index) for attribute in attributes)))
        )
        combined = pd.concat([attribute.loc[indices] for attribute in attributes], axis="columns")
        mask = combined.notna().all(axis="columns") \
            & combined.astype(str).apply(lambda column: column.str.strip().ne("")).all(axis="columns")

        return pd.concat([combined.loc[mask], self.loc[indices].loc[mask]], axis="columns")

In [ ]:
dataset_path = DATA_DIR / f"dataset.{K}.csv"

if dataset_path.exists():
    dataset = pd.read_csv(dataset_path, index_col=0)
    print(f"Loaded cached dataset from {dataset_path}")
else:
    lyrics = MusicSeries.from_tar(DATA_DIR / "processed_lyrics.tar.gz")
    genres = MusicSeries.from_csv(DATA_DIR / "id_genres.csv")
    tags = MusicSeries.from_csv(DATA_DIR / "id_tags.csv")
    info = MusicDataFrame.from_csv(DATA_DIR / "id_information.csv")
    audio_stats = MusicDataFrame.from_csv(DATA_DIR / "id_mfcc_stats.tsv.bz2")

    dataset = audio_stats.intersection(
        genres.top(K), tags,
        *[info[column] for column in info.columns],
        lyrics,
    )
    dataset.to_csv(dataset_path)
    print(f"Built and cached dataset to {dataset_path}")

print(f"Dataset shape: {dataset.shape}")
dataset.head()

## 2. Εξαγωγή Χαρακτηριστικών & Embeddings — Feature Extraction

We produce four separate embedding matrices, all indexed by Song ID:

### Text Embeddings (Lyrics)

> **Deviation from assignment**: The assignment suggests training Word2Vec or Doc2Vec on the lyrics corpus. We instead use a **pre-trained Sentence Transformer** (`all-MiniLM-L6-v2`), which maps each song's lyrics to a **384-dimensional** dense vector. This is a deliberate choice: the lyrics have already been stemmed and preprocessed, which removes inflection — a pre-trained transformer with BPE subword tokenization handles this gracefully, while training Word2Vec on stemmed text would yield a weaker vocabulary. The model and embedding dimensionality are **indicative**, not optimized — a larger model (e.g. `all-mpnet-base-v2` at 768 dims) could improve downstream quality.

### Audio Embeddings (MFCC)

> **Deviation from assignment**: Rather than PCA (the default suggestion), we train a **PyTorch autoencoder** (the bonus approach) to compress the 104-dimensional MFCC feature vector (13 means + 91 covariance entries) down to a bottleneck of **⌈√104⌉ = 11 dimensions**. The autoencoder uses a single hidden layer with SiLU activation. The bottleneck size is a simple heuristic — **not tuned** — and could be increased for better reconstruction fidelity.

### Genre & Tag Encodings

Genres and tags are both comma-separated multilabel strings. We apply binary one-hot encoding via `str.get_dummies(sep=",")`, producing a binary matrix where each column is a genre/tag and each row indicates presence (1) or absence (0).

In [ ]:
if torch.cuda.is_available():
    torch.set_default_device("cuda")


class AudioAutoencoder(torch.nn.Module):

    def __init__(self, input_dim: int, bottleneck: int | None = None) -> None:
        super().__init__()

        self.bottleneck = bottleneck or math.isqrt(input_dim) + 1

        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(input_dim, self.bottleneck),
            torch.nn.SiLU(),
        )
        self.decoder = torch.nn.Sequential(
            torch.nn.Linear(self.bottleneck, input_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))

    @staticmethod
    @torch.no_grad
    def normalize(data: torch.Tensor) -> torch.Tensor:
        return (data - data.mean(dim=0)) / data.std(dim=0)

    def compile(self,
        optimizer: torch.optim.Optimizer | None = None,
        loss_fn: torch.nn.MSELoss | None = None,
        lr: float = 1e-3,
    ) -> None:
        self.optimizer = optimizer or torch.optim.Adam(self.parameters(), lr=lr)
        self.loss_fn = loss_fn or torch.nn.MSELoss()

    def fit(self, data: torch.Tensor,
        epochs: int = 1,
        batch_size: int | None = None,
    ) -> None:
        dataset = torch.utils.data.TensorDataset(self.normalize(data))
        loader = torch.utils.data.DataLoader(dataset,
            batch_size=batch_size or math.isqrt(len(data)) + 1,
            shuffle=True,
            generator=torch.Generator(device=data.device),
        )

        self.train()

        with rich.progress.Progress(
            rich.progress.TextColumn("[bold blue]{task.description}"),
            rich.progress.BarColumn(),
            rich.progress.MofNCompleteColumn(),
            rich.progress.TextColumn("loss: {task.fields[loss]:.4f}"),
            rich.progress.TimeRemainingColumn(),
            rich.progress.TimeElapsedColumn(),
            refresh_per_second=60,
        ) as progress:
            epoch_task = progress.add_task("epoch", total=epochs, loss=0.)
            batch_task = progress.add_task("batch", total=len(loader), loss=0.)

            cumulative_loss = 0.

            for epoch in range(epochs):
                total_loss = 0.
                samples = 0

                progress.reset(batch_task, total=len(loader))

                for batch, in loader:
                    loss = self.loss_fn(self(batch), batch); self.optimizer.zero_grad()
                    loss.backward(); self.optimizer.step()

                    total_loss += loss.item() * batch.size(0); samples += batch.size(0)
                    progress.update(batch_task, advance=1, loss=total_loss / samples)

                cumulative_loss += total_loss / len(data)
                progress.update(epoch_task, advance=1, loss=cumulative_loss / (epoch + 1))

            progress.remove_task(batch_task); print()

        self.eval()

    @torch.no_grad
    def evaluate(self, data: torch.Tensor) -> float:
        normalized = self.normalize(data)
        return self.loss_fn(self(normalized), normalized).item()

    @torch.no_grad
    def encode(self, data: torch.Tensor) -> torch.Tensor:
        normalized = self.normalize(data)
        return self.encoder(normalized)


def encode_genres(genres: pd.Series) -> pd.DataFrame:
    return genres.str.get_dummies(sep=",")


def embed_audio(features: pd.DataFrame, epochs: int = 1) -> pd.DataFrame:
    tensor = torch.tensor(features.values, dtype=torch.float32)

    model = AudioAutoencoder(len(features.columns))
    model.compile()
    model.fit(tensor, epochs)
    embeddings = model.encode(tensor).numpy(force=True)

    return pd.DataFrame(embeddings,
        index=features.index,
        columns=[f"audio_{i:03d}" for i in range(model.bottleneck)],
    )


def embed_lyrics(lyrics: pd.Series, model_name: str = "all-MiniLM-L6-v2") -> pd.DataFrame:
    model = sentence_transformers.SentenceTransformer(model_name)
    embeddings = model.encode(lyrics.tolist())

    return pd.DataFrame(
        embeddings,
        index=lyrics.index,
        columns=[f"lyric_{i:03d}" for i in range(embeddings.shape[1])],
    )

In [ ]:
EPOCHS = 256

genres_path = DATA_DIR / f"dataset.{K}.genres.parquet"
audio_path  = DATA_DIR / f"dataset.{K}.audio.parquet"
lyrics_path = DATA_DIR / f"dataset.{K}.lyrics.parquet"
tags_path   = DATA_DIR / f"dataset.{K}.tags.parquet"

if all(p.exists() for p in (genres_path, audio_path, lyrics_path, tags_path)):
    genres_enc = pd.read_parquet(genres_path)
    audio_emb  = pd.read_parquet(audio_path)
    lyrics_emb = pd.read_parquet(lyrics_path)
    tags_enc   = pd.read_parquet(tags_path)
    print("Loaded cached embeddings.")
else:
    genres_enc = encode_genres(dataset["genres"])
    audio_emb  = embed_audio(
        dataset[[c for c in dataset.columns if c.startswith(("MFCC", "cov_"))]],
        EPOCHS,
    )
    lyrics_emb = embed_lyrics(dataset["lyrics"])
    tags_enc   = encode_genres(dataset["tags"])

    genres_enc.to_parquet(genres_path)
    audio_emb.to_parquet(audio_path)
    lyrics_emb.to_parquet(lyrics_path)
    tags_enc.to_parquet(tags_path)
    print("Generated and cached embeddings.")

print(f"Genre encodings:  {genres_enc.shape}")
print(f"Audio embeddings: {audio_emb.shape}")
print(f"Lyric embeddings: {lyrics_emb.shape}")
print(f"Tag encodings:    {tags_enc.shape}")

## 3. Οπτικοποίηση και Ανάλυση — Exploratory Data Analysis (EDA)

With the dataset loaded and all embeddings computed, we now explore the data through a series of visualizations. Each plot is produced by a standalone function that accepts an `EDAData` container holding the dataset and all embedding matrices.

In [ ]:
@dataclass
class EDAData:
    dataset: pd.DataFrame
    genres_enc: pd.DataFrame
    tags_enc: pd.DataFrame
    audio_emb: pd.DataFrame
    lyrics_emb: pd.DataFrame


data = EDAData(
    dataset=dataset,
    genres_enc=genres_enc,
    tags_enc=tags_enc,
    audio_emb=audio_emb,
    lyrics_emb=lyrics_emb,
)

### 3.1 Word Clouds by Genre

To pick the two genres for word cloud comparison, we automatically detect the **most different pair** among the top-5 genres. "Difference" is measured by cosine distance between genre-level tag profile vectors — i.e., for each genre we sum the binary tag vectors of all its songs (normalized), then find the pair with the lowest cosine similarity. This ensures the word clouds are maximally contrastive.

In [ ]:
def top_genre_names(data: EDAData, k: int = 5) -> list[str]:
    return data.genres_enc.sum().sort_values(ascending=False).head(k).index.tolist()


def most_different_genres(data: EDAData, genres: list[str]) -> tuple[str, str]:
    profiles = {}

    for genre in genres:
        mask = data.genres_enc[genre].astype(bool)
        profiles[genre] = data.tags_enc.loc[mask].sum().values.astype(float)
        profiles[genre] /= profiles[genre].sum() or 1.

    worst_sim = 1.
    pair = (genres[0], genres[1])

    for i, g1 in enumerate(genres):
        for g2 in genres[i + 1:]:
            sim = cosine_similarity(
                profiles[g1].reshape(1, -1),
                profiles[g2].reshape(1, -1),
            )[0, 0]

            if sim < worst_sim:
                worst_sim = sim
                pair = (g1, g2)

    return pair


def plot_wordcloud(data: EDAData, genre: str,
    ax: matplotlib.axes.Axes | None = None,
) -> matplotlib.figure.Figure:
    mask = data.genres_enc[genre].astype(bool)
    tag_freq = data.tags_enc.loc[mask].sum().to_dict()

    cloud = WordCloud(
        width=800, height=400,
        background_color="white",
    ).generate_from_frequencies(tag_freq)

    if ax is None: fig, ax = plt.subplots(figsize=(10, 5))
    else: fig = ax.figure

    ax.imshow(cloud, interpolation="bilinear")
    ax.set_title(f"Tag Word Cloud: {genre}")
    ax.axis("off")

    return fig


top_genres = top_genre_names(data, K)
g1, g2 = most_different_genres(data, top_genres)

print(f"Top {K} genres: {top_genres}")
print(f"Most different pair: {g1} vs {g2}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5))
plot_wordcloud(data, g1, ax=ax1)
plot_wordcloud(data, g2, ax=ax2)
fig.tight_layout()
plt.show()

### 3.2 Top Tags

A simple bar chart of the 10 most frequent user-generated tags across the entire dataset. Tags are crowd-sourced descriptors and often overlap with (but are not identical to) genre labels — they capture listener perception rather than editorial taxonomy.

In [ ]:
def plot_top_tags(data: EDAData, n: int = 10,
    ax: matplotlib.axes.Axes | None = None,
) -> matplotlib.figure.Figure:
    counts = data.tags_enc.sum().sort_values(ascending=True).tail(n)

    if ax is None: fig, ax = plt.subplots(figsize=(10, 6))
    else: fig = ax.figure

    ax.barh(counts.index, counts.values)
    ax.set_xlabel("Frequency")
    ax.set_title(f"Top {n} Most Frequent Tags")

    return fig


plot_top_tags(data)
plt.show()

### 3.3 Dimensionality Reduction — t-SNE

We apply t-SNE to project both audio and lyrics embeddings down to 2D, then plot each song as a point colored by its genre(s). Since genres are multilabel, a song may appear under multiple genre layers — we use low alpha (0.1) to handle overlap.

The side-by-side comparison reveals which modality separates genres more cleanly. We expect audio features to produce tighter clusters for genres with distinctive sonic profiles (e.g. electronic, hip-hop), while lyrics embeddings may better distinguish genres that share similar sound but differ thematically.

> **Note**: t-SNE on ~54k samples is CPU-bound via scikit-learn and takes a few minutes. Results are computed live.

In [ ]:
def reduce_tsne(embeddings: pd.DataFrame,
    n_components: int = 2,
    **kwargs,
) -> np.ndarray:
    return TSNE(
        n_components=n_components,
        random_state=42,
        **kwargs,
    ).fit_transform(embeddings.values)


def plot_scatter_by_genre(data: EDAData,
    modality: str,
    coords_2d: np.ndarray,
    top_k: int = 5,
    ax: matplotlib.axes.Axes | None = None,
) -> matplotlib.figure.Figure:
    if ax is None: fig, ax = plt.subplots(figsize=(10, 8))
    else: fig = ax.figure

    genres = top_genre_names(data, top_k)

    for genre in genres:
        mask = data.genres_enc[genre].astype(bool).values
        ax.scatter(
            coords_2d[mask, 0],
            coords_2d[mask, 1],
            label=genre, alpha=0.1, s=5,
        )

    ax.set_title(f"t-SNE: {modality.capitalize()} Embeddings by Genre")
    ax.legend(markerscale=5)

    return fig


audio_2d = reduce_tsne(data.audio_emb)
lyrics_2d = reduce_tsne(data.lyrics_emb)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
plot_scatter_by_genre(data, "audio", audio_2d, top_k=K, ax=ax1)
plot_scatter_by_genre(data, "lyrics", lyrics_2d, top_k=K, ax=ax2)
fig.suptitle("Audio vs Text Embeddings — Genre Separation Comparison")
fig.tight_layout()
plt.show()

### 3.4 Genre Variety per Song

Music is complex — a song might be pure "rock" or straddle "rock, indie rock, alternative rock". This histogram shows the distribution of how many genres each song belongs to, illustrating the multilabel nature of the dataset. Most songs have 1–3 genres, but some belong to many more.

In [ ]:
def plot_genre_count_histogram(data: EDAData,
    ax: matplotlib.axes.Axes | None = None,
) -> matplotlib.figure.Figure:
    genre_counts = data.dataset["genres"].str.count(",") + 1

    if ax is None: fig, ax = plt.subplots(figsize=(10, 6))
    else: fig = ax.figure

    bins = range(1, genre_counts.max() + 2)

    ax.hist(genre_counts, bins=bins, edgecolor="black", align="left")
    ax.set_xlabel("Number of Genres per Song")
    ax.set_ylabel("Number of Songs")
    ax.set_title("Genre Variety: How Many Genres Does Each Song Belong To?")
    ax.set_xticks(range(1, genre_counts.max() + 1))

    return fig


plot_genre_count_histogram(data)
plt.show()

### 3.5 Genre Distribution

How many songs belong to each of the top genres? Because genres are multilabel, a single song can contribute to multiple bars — the counts reflect genre *membership*, not exclusive assignment.

In [ ]:
def plot_genre_distribution(data: EDAData, n: int = 10,
    ax: matplotlib.axes.Axes | None = None,
) -> matplotlib.figure.Figure:
    counts = data.genres_enc.sum().sort_values(ascending=True).tail(n)

    if ax is None: fig, ax = plt.subplots(figsize=(10, 6))
    else: fig = ax.figure

    ax.barh(counts.index, counts.values)
    ax.set_xlabel("Number of Songs")
    ax.set_title(f"Top {n} Genres by Song Count")

    return fig


plot_genre_distribution(data, n=10)
plt.show()

### 3.6 Lyrics Length Distribution

Some songs have dense, wordy lyrics while others are minimal. We look at three measures:
1. **Character count** — raw text length
2. **Word count** — total tokens after splitting on whitespace
3. **Meaningful word count** — words remaining after removing English stopwords (via NLTK)

This gives a sense of the corpus's text density and how much "signal" the lyrics embeddings have to work with.

In [ ]:
def plot_lyrics_length(data: EDAData,
    lang: str = "english",
) -> matplotlib.figure.Figure:
    lyrics = data.dataset["lyrics"]
    words = lyrics.str.split()

    stop = set(stopwords.words(lang))

    char_counts = lyrics.str.len()
    word_counts = words.str.len()
    meaningful_counts = words.apply(lambda ws: sum(1 for w in ws if w not in stop))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].hist(char_counts, bins=50, edgecolor="black")
    axes[0].set_xlabel("Character Count")
    axes[0].set_ylabel("Number of Songs")
    axes[0].set_title("Lyrics Length (Characters)")

    axes[1].hist(word_counts, bins=50, edgecolor="black")
    axes[1].set_xlabel("Word Count")
    axes[1].set_title("Lyrics Length (Words)")

    axes[2].hist(meaningful_counts, bins=50, edgecolor="black")
    axes[2].set_xlabel("Meaningful Word Count")
    axes[2].set_title("Lyrics Length (Without Stopwords)")

    fig.tight_layout()

    return fig


plot_lyrics_length(data)
plt.show()

### 3.7 Sentiment Analysis by Genre

The assignment suggests using VADER for sentiment analysis. However, our lyrics are **already stemmed** — words like "happy" appear as "happi", "beautiful" as "beauti", etc. VADER relies on a hand-crafted lexicon of exact word forms, so stemmed text would largely miss its vocabulary.

> **Deviation**: We use **DistilBERT** via `transformers.pipeline("sentiment-analysis")` instead. DistilBERT uses BPE subword tokenization, which gracefully handles stemmed tokens by splitting them into recognized subwords. The model outputs a label (POSITIVE/NEGATIVE) and a confidence score, which we map to a [-1, +1] sentiment scale.

The violin plot below shows sentiment score distributions for each of the top-5 genres. We might expect pop songs to skew more positive and genres like rock or alternative to show broader distributions.

In [ ]:
def compute_sentiment(lyrics: pd.Series,
    batch_size: int = 64,
) -> pd.Series:
    device = 0 if torch.cuda.is_available() else -1

    classifier = pipeline("sentiment-analysis",  # type: ignore[call-overload]
        device=device,
        truncation=True,
        max_length=512,
    )

    results = classifier(lyrics.tolist(), batch_size=batch_size)

    scores = pd.Series(
        [r["score"] if r["label"] == "POSITIVE" else -r["score"] for r in results],
        index=lyrics.index,
        name="sentiment",
    )

    return scores


def plot_sentiment_by_genre(data: EDAData,
    top_k: int = 5,
) -> matplotlib.figure.Figure:
    sentiment = compute_sentiment(data.dataset["lyrics"])
    genres = top_genre_names(data, top_k)

    rows = []

    for genre in genres:
        mask = data.genres_enc[genre].astype(bool)

        for score in sentiment.loc[mask]:
            rows.append({"genre": genre, "sentiment": score})

    expanded = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(10, 6))

    sns.violinplot(data=expanded, x="genre", y="sentiment", ax=ax)

    ax.set_xlabel("Genre")
    ax.set_ylabel("Sentiment Score")
    ax.set_title("Sentiment Distribution by Genre (DistilBERT)")
    ax.axhline(0, color="gray", linestyle="--", alpha=0.5)

    return fig


plot_sentiment_by_genre(data, top_k=K)
plt.show()

### 3.8 Similarity Analysis

Given a query song, we find its top-5 most similar songs by **cosine similarity** on two separate embedding spaces:
- **Lyrics similarity** — songs with thematically similar text
- **Audio similarity** — songs with similar acoustic profiles (MFCC-derived)

The two modalities often disagree: a song may sound similar to others in a completely different genre but share lyrical themes with songs from yet another genre. We showcase several examples from different genres to illustrate this.

In [ ]:
def find_similar_songs(data: EDAData, song_id: str,
    k: int = 5,
    modality: str = "both",
) -> pd.DataFrame:
    results = {}

    if modality in ("lyrics", "both"):
        query = data.lyrics_emb.loc[[song_id]]
        sims = cosine_similarity(query, data.lyrics_emb)[0]
        sims = pd.Series(sims, index=data.lyrics_emb.index, name="lyrics_sim")
        sims = sims.drop(song_id).sort_values(ascending=False).head(k)
        results["lyrics_sim"] = sims

    if modality in ("audio", "both"):
        query = data.audio_emb.loc[[song_id]]
        sims = cosine_similarity(query, data.audio_emb)[0]
        sims = pd.Series(sims, index=data.audio_emb.index, name="audio_sim")
        sims = sims.drop(song_id).sort_values(ascending=False).head(k)
        results["audio_sim"] = sims

    all_ids = set()

    for s in results.values():
        all_ids.update(s.index)

    info = data.dataset.loc[list(all_ids), ["song", "artist", "genres"]]

    for name, sims in results.items():
        info[name] = sims

    return info.sort_values(
        by=list(results.keys()),
        ascending=False,
    )


def display_similarity_results(data: EDAData, song_id: str,
    k: int = 5,
) -> None:
    song = data.dataset.loc[song_id]

    print(f"\nQuery: {song['artist']} — {song['song']} [{song['genres']}]\n")

    for modality in ("lyrics", "audio"):
        result = find_similar_songs(data, song_id, k, modality)

        print(f"Top {k} by {modality} similarity:")
        print(result[["song", "artist", "genres", f"{modality}_sim"]].to_string())
        print()

In [ ]:
# Showcase similarity search with songs from different genres
example_ids = [
    "9epP1yOXfLKlkR3S",  # Carolina Liar — I'm Not Over [rock, indie rock]
    "MtxPSiYt0J3CTojA",  # Whitney Houston — You're Still My Man [pop, soul]
    "LQ5D2Q5SE914jB8V",  # Yuno — Grapefruit [electronic]
    "RLGHhOE3qeWQUFey",  # Gin Blossoms — Allison Road [alternative rock, rock]
]

for song_id in example_ids:
    display_similarity_results(data, song_id)
    print("=" * 80)

## Observations

**Dataset**: By treating genres as multilabel, we retain ~54,000 songs — significantly more than the ~10,000 expected from a multiclass approach. Most songs belong to 1–3 genres, confirming that multilabel is the natural representation.

**Embeddings**: The audio autoencoder compresses 104 MFCC features down to 11 dimensions, while the sentence transformer produces 384-dimensional lyrics vectors. Both are compact representations chosen for convenience — larger models and wider bottlenecks would likely improve downstream tasks.

**t-SNE**: The audio embeddings show some genre-level clustering (electronic songs tend to group together), but significant overlap remains — unsurprising given the low-dimensional bottleneck. Lyrics embeddings show less distinct clusters, suggesting that thematic content alone is a weaker genre discriminator than acoustic features.

**Sentiment**: DistilBERT sentiment distributions are fairly similar across genres, with most genres showing a slight positive skew. This is consistent with the observation that pop lyrics tend toward positive sentiment, while rock and alternative show broader variance.

**Similarity**: Audio and lyrics similarity often retrieve very different neighbors for the same query — a song's closest acoustic match may come from a completely different genre than its closest thematic match. This motivates multimodal fusion for downstream classification tasks.

## Part B Preview — Multi-label Classification and Fusion

For Part B we keep the multilabel representation: every song is mapped to a set of genre labels, encoded as binary switches. The classification pipeline compares text-only, audio-only, concatenated early fusion, late fusion, and a bilinear pooling ablation.

**Standard early fusion** concatenates the lyric and audio embeddings. **Bilinear pooling**, also called **outer-product fusion**, instead forms every pairwise text-audio product `text_i * audio_j` and uses only these cross-modal interaction features. With 384 lyric dimensions and 11 audio dimensions, the bilinear representation has `384 * 11 = 4224` features per song.

This ablation asks whether interaction information alone is useful. It is not a correlation matrix; it is a per-song outer-product feature map. Because its dimensionality is much larger than the audio-only or concatenated baselines, the logistic models use fixed representation-specific regularization values: `C=10` for audio, `C=1` for text and concatenated early fusion, and `C=0.1` for bilinear pooling. The 10-fold cross-validation requested in the assignment is used to evaluate these fixed configurations, not to perform automatic hyperparameter tuning.


In [ ]:
from src.classification import run_experiments, plot_f1_comparison, plot_label_confusions

# Full Part B run with the fixed representation-specific C defaults:
# audio C=10, text C=1, concatenated early fusion C=1, bilinear pooling C=0.1
# results = run_experiments(DATA_DIR, k=K, n_splits=10)

# Quicker smoke run while developing the notebook:
# results = run_experiments(DATA_DIR, k=K, n_splits=3, max_samples=1000, include_clustering=False)

# results.metrics
# plot_f1_comparison(results.metrics)
# plot_label_confusions(results.confusion_matrices["Bilinear pooling"], "Bilinear Pooling")